In [1]:
!pip install albumentations segmentation-models-pytorch scikit-image scipy tqdm

In [2]:
import subprocess
subprocess.run(["pip", "install", "albumentations", 
                "segmentation-models-pytorch", 
                "scikit-image", "scipy", "tqdm"], check=True)

CompletedProcess(args=['pip', 'install', 'albumentations', 'segmentation-models-pytorch', 'scikit-image', 'scipy', 'tqdm'], returncode=0)

In [4]:
# ============================================================
#  STEP 1: Install dependencies (run this cell once, then restart kernel)
# ============================================================
# !pip install albumentations segmentation-models-pytorch scikit-image scipy tqdm

# ============================================================
#  STEP 2: Imports
# ============================================================
import os, glob, random, warnings
import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore")

# ============================================================
#  STEP 3: CONFIG  — only change DATA_ROOT if needed
# ============================================================
DATA_ROOT  = r'./Dataset_BUSI_with_GT'
SAVE_DIR   = './checkpoints'
EPOCHS     = 50
BATCH_SIZE = 12
LR         = 0.03
IMG_SIZE   = 512
SEED       = 42

# ============================================================
#  STEP 4: Reproducibility + Device
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ============================================================
#  STEP 5: Verify dataset path
# ============================================================
assert os.path.isdir(DATA_ROOT), f"❌ Folder not found: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, 'benign')),    "❌ 'benign' subfolder missing"
assert os.path.isdir(os.path.join(DATA_ROOT, 'malignant')), "❌ 'malignant' subfolder missing"
print(f"Dataset: {DATA_ROOT}  ✓")

# ============================================================
#  STEP 6: Dataset class
# ============================================================
class BUSIDataset(Dataset):
    """
    Loads images + masks from benign/ and malignant/ folders.
    Multiple masks per image are merged with OR (as done in the paper).
    """
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        # load RGB image
        image = np.array(Image.open(img_path).convert("RGB"))

        # find all mask files for this image and merge them
        base       = img_path.replace(".png", "")
        mask_files = sorted(glob.glob(base + "_mask*.png"))
        merged     = np.zeros(image.shape[:2], dtype=np.uint8)
        for mf in mask_files:
            m      = np.array(Image.open(mf).convert("L"))
            merged = np.clip(merged + (m > 127).astype(np.uint8), 0, 1)

        if self.transform:
            out   = self.transform(image=image, mask=merged)
            image = out["image"]           # float32 tensor [3, H, W]
            mask  = out["mask"].long()     # long tensor    [H, W]
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask  = torch.from_numpy(merged).long()

        return image, mask


# ============================================================
#  STEP 7: Helper — collect image paths + 80/20 split
# ============================================================
def get_image_paths(root):
    """Return all original image paths (no _mask files) from benign + malignant."""
    paths = []
    for cls in ["benign", "malignant"]:
        folder = os.path.join(root, cls)
        pngs   = sorted(glob.glob(os.path.join(folder, "*.png")))
        paths += [p for p in pngs if "_mask" not in os.path.basename(p)]
    return paths


def split_80_20(paths, seed=SEED):
    """Stratified 80/20 split keeping benign/malignant ratio."""
    benign    = sorted([p for p in paths if os.sep + "benign"    + os.sep in p])
    malignant = sorted([p for p in paths if os.sep + "malignant" + os.sep in p])

    rng = random.Random(seed)
    rng.shuffle(benign)
    rng.shuffle(malignant)

    nb = int(len(benign)    * 0.8)
    nm = int(len(malignant) * 0.8)

    train = benign[:nb] + malignant[:nm]
    val   = benign[nb:] + malignant[nm:]
    return train, val


# build datasets & loaders
all_paths             = get_image_paths(DATA_ROOT)
train_paths, val_paths = split_80_20(all_paths)

print(f"\nTotal images : {len(all_paths)}")
print(f"Train        : {len(train_paths)}")
print(f"Val          : {len(val_paths)}")

# ============================================================
#  STEP 8: Augmentations (paper Section 3.3)
# ============================================================
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.GaussNoise(p=0.3),
    A.Affine(
        scale           = (0.9, 1.1),
        translate_percent = 0.05,
        rotate          = (-10, 10),
        shear           = (-5, 5),
        p               = 0.5
    ),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std =(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_dataset = BUSIDataset(train_paths, transform=train_transform)
val_dataset   = BUSIDataset(val_paths,   transform=val_transform)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")

# ============================================================
#  STEP 9: Loss  —  CE + SoftDice  (alpha=beta=1, paper eq.10)
# ============================================================
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)[:, 1]   # foreground probability
        t     = targets.float()
        num   = 2.0 * (probs * t).sum()
        den   = probs.pow(2).sum() + t.pow(2).sum()
        return 1.0 - (num + self.smooth) / (den + self.smooth)


class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss()
        self.dice = SoftDiceLoss()

    def forward(self, logits, targets):
        return self.ce(logits, targets) + self.dice(logits, targets)


# ============================================================
#  STEP 10: Metrics  (paper Section 3.2)
#           Dice, Jaccard, Precision, Sensitivity, Specificity,
#           ASSD, HD  — all computed at image level then averaged
# ============================================================
def compute_metrics(preds_list, masks_list):
    dices, jaccs  = [], []
    precs, senss  = [], []
    specs         = []
    assds, hds    = [], []

    for pred, mask in zip(preds_list, masks_list):
        pred = pred.astype(bool)
        mask = mask.astype(bool)

        TP = int(( pred &  mask).sum())
        FP = int(( pred & ~mask).sum())
        TN = int((~pred & ~mask).sum())
        FN = int((~pred &  mask).sum())

        dices.append((2*TP) / (2*TP + FP + FN + 1e-8))
        jaccs.append(    TP / (  TP + FP + FN + 1e-8))
        precs.append(    TP / (  TP + FP      + 1e-8))
        senss.append(    TP / (  TP + FN      + 1e-8))
        specs.append(    TN / (  TN + FP      + 1e-8))

        # ASSD & HD: require at least one foreground pixel in both maps
        if pred.any() and mask.any():
            pred_edge = pred & ~binary_erosion(pred)
            mask_edge = mask & ~binary_erosion(mask)

            if pred_edge.any() and mask_edge.any():
                d_pred2mask = distance_transform_edt(~mask)[pred_edge]
                d_mask2pred = distance_transform_edt(~pred)[mask_edge]

                assd = (d_pred2mask.sum() + d_mask2pred.sum()) / \
                       (len(d_pred2mask)  + len(d_mask2pred))
                hd   = max(d_pred2mask.max(), d_mask2pred.max())
            else:
                assd, hd = np.nan, np.nan
        else:
            assd, hd = np.nan, np.nan

        assds.append(assd)
        hds.append(hd)

    return {
        "Dice (%)":        np.mean(dices)    * 100,
        "Jaccard (%)":     np.mean(jaccs)    * 100,
        "Precision (%)":   np.mean(precs)    * 100,
        "Sensitivity (%)": np.mean(senss)    * 100,
        "Specificity (%)": np.mean(specs)    * 100,
        "ASSD":            np.nanmean(assds),
        "HD":              np.nanmean(hds),
    }


# ============================================================
#  STEP 11: LR schedule  — poly decay (paper eq.18)
# ============================================================
def poly_lr(optimizer, init_lr, cur_iter, total_iter, power=0.9):
    lr = init_lr * ((1.0 - cur_iter / total_iter) ** power)
    for pg in optimizer.param_groups:
        pg["lr"] = max(lr, 1e-6)


# ============================================================
#  STEP 12: Model  — FPN + ResNet34 encoder
# ============================================================
model = smp.FPN(
    encoder_name    = "resnet34",
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = 2,          # background=0, tumor=1
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel      : FPN + ResNet34 encoder")
print(f"Parameters : {n_params/1e6:.2f} M   (paper reports ~23.16 M)")

optimizer = optim.SGD(model.parameters(),
                      lr=LR, momentum=0.9, weight_decay=5e-4)
criterion = CombinedLoss()

os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
#  STEP 13: Training loop
# ============================================================
total_iters  = EPOCHS * len(train_loader)
global_iter  = 0
best_dice    = 0.0
best_epoch   = 0
best_metrics = {}

print("\n" + "="*70)
print("Starting training ...")
print("="*70)

for epoch in range(1, EPOCHS + 1):

    # ── train ────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for imgs, masks in tqdm(train_loader,
                            desc=f"Epoch {epoch:3d}/{EPOCHS} [train]",
                            leave=False):
        imgs,  masks = imgs.to(device), masks.to(device)
        poly_lr(optimizer, LR, global_iter, total_iters)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        train_loss  += loss.item()
        global_iter += 1

    train_loss /= len(train_loader)

    # ── validate ─────────────────────────────────────────────
    model.eval()
    val_loss   = 0.0
    all_preds  = []
    all_masks  = []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader,
                                desc=f"Epoch {epoch:3d}/{EPOCHS} [val]  ",
                                leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            logits      = model(imgs)
            val_loss   += criterion(logits, masks).item()

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = (probs > 0.5).cpu().numpy()
            all_preds.extend(preds)
            all_masks.extend(masks.cpu().numpy())

    val_loss /= len(val_loader)
    m = compute_metrics(all_preds, all_masks)

    # print every epoch
    print(f"Ep {epoch:3d} | "
          f"TL={train_loss:.4f} VL={val_loss:.4f} | "
          f"Dice={m['Dice (%)']:5.2f}% "
          f"Jacc={m['Jaccard (%)']:5.2f}% "
          f"Prec={m['Precision (%)']:5.2f}% "
          f"Sens={m['Sensitivity (%)']:5.2f}% "
          f"Spec={m['Specificity (%)']:5.2f}% "
          f"ASSD={m['ASSD']:6.2f} "
          f"HD={m['HD']:6.2f}")

    # save best
    if m["Dice (%)"] > best_dice:
        best_dice    = m["Dice (%)"]
        best_epoch   = epoch
        best_metrics = m.copy()
        ckpt_path    = os.path.join(SAVE_DIR, "best_fpn_busi.pth")
        torch.save(model.state_dict(), ckpt_path)
        print(f"  ✓ Best model saved  →  {ckpt_path}  (Dice={best_dice:.2f}%)")

# ============================================================
#  STEP 14: Final report vs Table 1
# ============================================================
PAPER_FPN = {
    "Dice (%)":        74.20,
    "Jaccard (%)":     65.96,
    "Precision (%)":   77.30,
    "Sensitivity (%)": 75.38,
    "Specificity (%)": 97.80,
    "ASSD":            22.61,
    "HD":              78.08,
}

print("\n" + "="*60)
print(f"  TABLE 1 COMPARISON  —  Best epoch: {best_epoch}")
print("="*60)
print(f"  {'Metric':<20} {'Ours':>10}  {'Paper':>10}")
print("-"*60)
for k in PAPER_FPN:
    ours  = best_metrics.get(k, float('nan'))
    paper = PAPER_FPN[k]
    diff  = ours - paper
    flag  = "▲" if diff >= 0 else "▼"
    print(f"  {k:<20} {ours:>10.2f}  {paper:>10.2f}   {flag}{abs(diff):.2f}")
print("="*60)
print("\nDone. Checkpoint saved to:", os.path.join(SAVE_DIR, "best_fpn_busi.pth"))

Device : cuda
GPU    : NVIDIA A100-SXM4-80GB
Dataset: ./Dataset_BUSI_with_GT  ✓

Total images : 647
Train        : 517
Val          : 130

Train batches : 44
Val   batches : 11

Model      : FPN + ResNet34 encoder
Parameters : 23.16 M   (paper reports ~23.16 M)

Starting training ...


Ep   1 | TL=0.8482 VL=0.6648 | Dice=34.79% Jacc=27.54% Prec=48.16% Sens=33.86% Spec=99.09% ASSD= 46.81 HD=122.54
  ✓ Best model saved  →  ./checkpoints/best_fpn_busi.pth  (Dice=34.79%)


Ep   2 | TL=0.5565 VL=0.4917 | Dice=57.33% Jacc=47.70% Prec=72.97% Sens=55.53% Spec=99.08% ASSD= 31.39 HD=110.21
  ✓ Best model saved  →  ./checkpoints/best_fpn_busi.pth  (Dice=57.33%)


Ep   3 | TL=0.5608 VL=0.4992 | Dice=58.51% Jacc=47.68% Prec=63.13% Sens=64.67% Spec=96.97% ASSD= 34.65 HD=113.85
  ✓ Best model saved  →  ./checkpoints/best_fpn_busi.pth  (Dice=58.51%)


Ep   4 | TL=0.4302 VL=0.4613 | Dice=58.61% Jacc=48.05% Prec=73.63% Sens=55.21% Spec=98.32% ASSD= 25.44 HD= 99.15
  ✓ Best model saved  →  ./checkpoints/best_fpn_busi.pth  (Dice=58.61%)


...

Ep  46 | TL=0.1514 VL=0.2846 | Dice=78.15% Jacc=69.43% Prec=81.04% Sens=80.79% Spec=97.52% ASSD= 17.00 HD= 61.87
  ✓ Best model saved  →  ./checkpoints/best_fpn_busi.pth  (Dice=78.15%)


Ep  47 | TL=0.1460 VL=0.2818 | Dice=78.13% Jacc=69.64% Prec=83.10% Sens=79.32% Spec=97.91% ASSD= 15.62 HD= 59.81


Ep  48 | TL=0.1461 VL=0.2803 | Dice=79.23% Jacc=70.58% Prec=82.99% Sens=80.23% Spec=97.93% ASSD= 14.02 HD= 56.16
  ✓ Best model saved  →  ./checkpoints/best_fpn_busi.pth  (Dice=79.23%)


Ep  49 | TL=0.1377 VL=0.2776 | Dice=79.26% Jacc=70.62% Prec=83.58% Sens=79.95% Spec=97.92% ASSD= 14.36 HD= 56.86
  ✓ Best model saved  →  ./checkpoints/best_fpn_busi.pth  (Dice=79.26%)


Ep  50 | TL=0.1387 VL=0.2840 | Dice=78.71% Jacc=70.23% Prec=83.99% Sens=79.13% Spec=97.95% ASSD= 14.60 HD= 57.21

  TABLE 1 COMPARISON  —  Best epoch: 49
  Metric                     Ours       Paper
------------------------------------------------------------
  Dice (%)                  79.26       74.20   ▲5.06
  Jaccard (%)               70.62       65.96   ▲4.66
  Precision (%)             83.58       77.30   ▲6.28
  Sensitivity (%)           79.95       75.38   ▲4.57
  Specificity (%)           97.92       97.80   ▲0.12
  ASSD                      14.36       22.61   ▼8.25
  HD                        56.86       78.08   ▼21.22

Done. Checkpoint saved to: ./checkpoints/best_fpn_busi.pth
